<a href="https://colab.research.google.com/github/Dev1ze32/llma3.2_3b_model_test/blob/main/content/02-getting-started/jupyter_notebooks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

# 1. This triggers an interactive input box.
# Paste your token here when prompted and hit enter.
login()

model_id = "meta-llama/Llama-3.2-3B-Instruct"

# Configure 4-bit quantization to fit safely within Colab's free T4 GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Loading model (this will take a few minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# Initialize the text generation pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

# Format your prompt using Llama 3.2's chat template structure
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Write a quick Python function to check if a word is a palindrome."}
]

print("Running inference...\n")
outputs = pipe(messages, max_new_tokens=512)

print("Result:")
print(outputs[0]["generated_text"][-1]["content"])

In [3]:
import os
import torch
from PyPDF2 import PdfReader
from sentence_transformers import SentenceTransformer, util

def extract_and_chunk_pdf(file_path, chunk_size=300):
    """
    Reads a PDF and splits it into smaller chunks of text.
    Chunking is necessary because models have a max token limit (512 for E5-base).
    """
    print(f"Reading {file_path}...")
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        extracted = page.extract_text()
        if extracted:
            text += extracted + " "

    # Simple word-based chunking
    words = text.split()
    chunks = [" ".join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size)]
    return chunks

def main():
    pdf_path = "handbook.pdf"

    if not os.path.exists(pdf_path):
        print(f"Error: {pdf_path} not found in the current directory.")
        return

    # 1. Load Knowledge Base
    raw_passages = extract_and_chunk_pdf(pdf_path, chunk_size=300)

    # 2. Load the E5 Model
    print("Loading intfloat/multilingual-e5-base...")
    model = SentenceTransformer('intfloat/multilingual-e5-base')

    # 3. Format and Encode Passages
    # E5 REQUIREMENT: Documents must be prefixed with "passage: "
    formatted_passages = [f"passage: {text}" for text in raw_passages]

    print(f"Encoding {len(formatted_passages)} chunks into vector embeddings...")
    # normalize_embeddings=True is recommended for cosine similarity
    passage_embeddings = model.encode(
        formatted_passages,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    print("\nKnowledge Base Ready!\n")

    # 4. Interactive Search Loop
    while True:
        user_query = input("Ask a question about the handbook (or type 'quit'): ")
        if user_query.lower() in ['quit', 'exit', 'q']:
            break

        # E5 REQUIREMENT: Search queries must be prefixed with "query: "
        formatted_query = f"query: {user_query}"

        # Encode the query
        query_embedding = model.encode(
            formatted_query,
            convert_to_tensor=True,
            normalize_embeddings=True
        )

        # 5. Calculate Cosine Similarity to find the best matches
        hits = util.semantic_search(query_embedding, passage_embeddings, top_k=3)[0]

        print("\n" + "="*50)
        print("🔍 TOP 3 SEARCH RESULTS:")
        print("="*50)
        for i, hit in enumerate(hits, 1):
            score = hit['score']
            chunk_text = raw_passages[hit['corpus_id']]
            print(f"\n--- Result {i} (Confidence Score: {score:.4f}) ---")
            print(f"{chunk_text}...\n")

if __name__ == "__main__":
    main()

Error: handbook.pdf not found in the current directory.


In [2]:
pip install pypdf2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 9.2 MB/s eta 0:00:00
